In [1]:
import os
import random
import gc
import argparse
import numpy as np
import h5py as h5
import torch
import torch.nn.functional as F
import torch.nn as nn
import wandb
from models.autoencoder import Autoencoder 
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from scipy.stats import binned_statistic

In [2]:
#setting a seed like in ae_legacy
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
def print_h5_tree(h, prefix=""):
    for k in h.keys():
        item = h[k]
        if hasattr(item, "keys"):
            print(prefix + f"[GROUP] {k}")
            print_h5_tree(item, prefix + "  ")
        else:
            try:
                print(prefix + f"{k}: shape={item.shape}, dtype={item.dtype}")
            except Exception:
                print(prefix + f"{k}: <dataset>")

In [5]:
def fit_standard_scaler(X, eps=1e-8):
    mu  = X.mean(axis=0).astype(np.float32)
    std = X.std(axis=0).astype(np.float32)
    std = np.where(std < eps, 1.0, std)
    return mu, std

def transform_standard(X, mu, std):
    return (X - mu) / (std + 1e-8)

In [6]:
class PerSampleMSE(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss(reduction="none")
    def forward(self, recon, target):
        per_feat = self.mse(recon, target)
        return per_feat.mean(dim=1)

In [7]:
def inference(ae, Xz, loss_fn, device, batch_size=4096):
    ae.eval()
    n = Xz.shape[0]
    out = np.empty(n, dtype=np.float32)
    with torch.no_grad():
        for i0 in range(0, n, batch_size):
            i1 = min(i0 + batch_size, n)
            xb = torch.tensor(Xz[i0:i1], dtype=torch.float32, device=device)
            recon, _ = ae(xb)
            loss_b = loss_fn(recon, xb)
            out[i0:i1] = loss_b.detach().cpu().numpy()
    return out

In [8]:
def abcd_counts(loss_1, loss_2, percent_1, percent_2):
    thresh_1 = np.quantile(loss_1, percent_1)
    thresh_2 = np.quantile(loss_2, percent_2)
    A = int(((loss_1 > thresh_1) & (loss_2 > thresh_2)).sum())
    B = int(((loss_1 > thresh_1) & (loss_2 <= thresh_2)).sum())
    C = int(((loss_1 <= thresh_1) & (loss_2 > thresh_2)).sum())
    D = int(((loss_1 <= thresh_1) & (loss_2 <= thresh_2)).sum())
    return thresh_1, thresh_2, A, B, C, D

In [9]:
def nonclosure_A(A, B, C, D, eps=1e-8):
    A_hat = (B * C) / max(D, eps)
    if A_hat <= 0:
        return np.inf, A_hat
    return (A - A_hat) / A_hat, A_hat

In [4]:
def profile_plot(ax, x, y, nbins=30, logx=False, min_per_bin=20, label="mean ± SE"):
    x = np.asarray(x)
    y = np.asarray(y)
    m = np.isfinite(x) & np.isfinite(y)
    if logx:
        m &= (x > 0)

    x = x[m]
    y = y[m]

    # bin along x (linear or log space)
    if logx:
        xu = np.log10(x)
    else:
        xu = x

    # uniform bins over the chosen coordinate
    lo = float(xu.min())
    hi = float(xu.max())
    if lo == hi:
        hi = np.nextafter(hi, np.inf)

    edges = np.linspace(lo, hi, nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    # stats per bin
    mean, _, _ = binned_statistic(xu, y, statistic="mean", bins=edges)
    std,  _, _ = binned_statistic(xu, y, statistic="std",  bins=edges)
    cnt,  _, _ = binned_statistic(xu, y, statistic="count", bins=edges)

    sem = std / np.sqrt(np.maximum(cnt, 1))

    # keep well populated bins
    good = cnt >= min_per_bin
    xc = centers[good]
    ym = mean[good]
    ye = sem[good]

    # convert x axis back from log if needed
    if logx:
        xplot = 10.0 ** xc
        ax.set_xscale("log")
    else:
        xplot = xc

    ax.errorbar(xplot, ym, yerr=ye, fmt="o", ms=3, lw=1, capsize=2, label=label)
    ax.grid(alpha=0.3)
    return {"x": xplot, "mean": ym, "sem": ye, "count": cnt[good]}

In [11]:
def run(config):
    set_seed(config.get("seed", 123))

    #device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using: {device}", flush=True)

    print("Logging in to wandb...", flush=True)
    wandb.login(key="24d1d60ce26563c74d290d7b487cb104fc251271")
    wandb.init(project="AE vs. Contrastive ABCD",
               settings=wandb.Settings(_disable_stats=True),
               config=config)
    run_name = wandb.run.name
    print(f"Run name: {run_name}", flush=True)

    #data 
    data_path = config["data_path"]
    sig_path = config["sig_path"]

    #contrastive outputs
    contrast_bkg_scores = config["contrast_bkg_scores"]
    contrast_sig_scores = config["contrast_sig_scores"]

    #load the background data
    print("Loading background dataset...", flush=True)
    with h5.File(data_path, "r") as f:
        r = f["data"] if "data" in f else f

        print("H5 tree (bkg):")
        print_h5_tree(r)

        x_train = r["Background_data"]["Train"]["DATA"][:]
        x_test = r["Background_data"]["Test"]["DATA"][:]
        print(f"Train shape: {x_train.shape}, Test shape: {x_test.shape}", flush=True)

    #load the signal data
    print("Loading signal dataset...", flush=True)
    with h5.File(sig_path, "r") as f:
        r = f["data"] if "data" in f else f

        print("H5 tree (sig):")
        print_h5_tree(r)

        #this is janky maybe fix one day (but preprocessed sig that same as bkgd)
        sig_train = r["Background_data"]["Train"]["DATA"][:]
        sig_test = r["Background_data"]["Test"]["DATA"][:]
        Xsig_raw = np.concatenate([sig_train, sig_test], axis=0)
        print(f"Signal total shape: {Xsig_raw.shape}", flush=True)

    #cleans up pre processing with some padding
    def zero_out_padding(X):
        X = X.copy()
        pad = (X == 0.0).all(axis=-1)
        X[pad]= 0.0
        return X

    Xtr_raw = zero_out_padding(x_train)
    Xte_raw = zero_out_padding(x_test)
    Xsig_raw = zero_out_padding(Xsig_raw)

    #flatten 
    def flatten(x):
        n, nobj, fdim = x.shape
        return x.reshape(n, nobj*fdim)

    X1_train_raw = flatten(Xtr_raw)
    X1_test_raw = flatten(Xte_raw)
    X1_sig_raw = flatten(Xsig_raw)

    #standardize
    mu1, std1 = fit_standard_scaler(X1_train_raw)
    X1_train_z = transform_standard(X1_train_raw, mu1, std1)
    X1_test_z = transform_standard(X1_test_raw, mu1, std1)
    X1_sig_z = transform_standard(X1_sig_raw, mu1, std1)

    axis2_bkg = np.load(contrast_bkg_scores).astype(np.float32)
    axis2_sig = np.load(contrast_sig_scores).astype(np.float32)

    #check contrastive and bkg array have same length!!
    if len(axis2_bkg) != len(X1_test_raw):
        raise ValueError("Contrastive length mismatch (bkg)")
    if len(axis2_sig) != len(X1_sig_raw):
        raise ValueError("Contrastive length mismatch (sig)")

    # z-score contrastive scores using background test statistics
    mu = float(axis2_bkg.mean())
    std = float(axis2_bkg.std() + 1e-8)
    axis2_bkg = (axis2_bkg - mu)/std
    axis2_sig = (axis2_sig - mu)/std

    #build the AE
    feat = X1_train_z.shape[1]
    reco_loss_fn = PerSampleMSE().to(device)

    ae_cfg = {
        "features": feat,
        "latent_dim": config["ae_latent"],
        "encoder_config": {"nodes": config["enc_nodes"]},
        "decoder_config": {"nodes": config["dec_nodes"] + [feat]},
        "alpha": config["alpha"]
    }

    ae = Autoencoder(ae_cfg).to(device)
    optimizer = torch.optim.Adam(ae.parameters(), lr=float(config["ae_lr"]))

    epochs = config["epochs"]
    batch_size = config["batch_size"]

    X1 = torch.tensor(X1_train_z, dtype=torch.float32, device=device)

    #training AE
    print("Starting AE training...", flush=True)
    for epoch in range(config["epochs"]):
        perm = torch.randperm(len(X1))
        losses = []

        for i0 in range(0, len(X1), config["batch_size"]):
            idx = perm[i0:i0+config["batch_size"]]
            xb = X1[idx]

            recon, _ = ae(xb)
            loss = reco_loss_fn(recon, xb).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        avg = np.mean(losses)

        wandb.log({"AE_loss": avg, "epoch": epoch})
        print(f"Epoch {epoch}: {avg:.6f}")

    #save the AE
    ae_path = os.path.join(outdir, "ae_axis1.pth")
    torch.save(ae.state_dict(), ae_path)
    wandb.save(ae_path)
    print("Saved AE:", ae_path, flush=True)

    #inference
    reco_test = inference(ae, X1_test_z, reco_loss_fn, device)
    reco_sig = inference(ae, X1_sig_z, reco_loss_fn, device) 

    #mask finite
    mask_bkg = np.isfinite(reco_test) & np.isfinite(axis2_bkg) & (reco_test > 0)
    mask_sig = np.isfinite(reco_sig) & np.isfinite(axis2_sig) & (reco_sig > 0)

    reco_test_m = reco_test[mask_bkg]
    axis2_bkg_m = axis2_bkg[mask_bkg]
    reco_sig_m = reco_sig[mask_sig]
    axis2_sig_m = axis2_sig[mask_sig]
    
    #ABCD scan
    best = {"nonclosure": np.inf}
    percent = np.linspace(0.75, 0.98, 24)

    for p1 in percent:
        for p2 in percent:
            t1, t2, A, B, C, D = abcd_counts(reco_test, axis2_bkg, p1, p2)
            if (A < min_A) or (D < min_D):
                continue
            nc, A_hat = nonclosure_A(A, B, C, D)
            if np.isfinite(nc) and abs(nc) < abs(best["nonclosure"]):
                best.update(dict(p1=p1, p2=p2, t1=t1, t2=t2, A=A, B=B, C=C, D=D,
                                 A_hat=A_hat, nonclosure=nc))
    t1_opt = best["t1"]
    t2_opt = best["t2"]

    print(f"Optimized p1={best.get('p1'):.3f}, p2={best.get('p2'):.3f}", flush=True)
    print(f"Optimized thresholds: t1={t1_opt:.6g}, t2={t2_opt:.6g}", flush=True)
    print(f"Nonclosure: {100.0*best['nonclosure']:.3f}%", flush=True)

    #log all the optimized thresholds and nonclosure and ABCD
    wandb.log({
        "ABCD/opt_p1": best.get("p1", -1),
        "ABCD/opt_p2": best.get("p2", -1),
        "ABCD/opt_t1": float(t1_opt),
        "ABCD/opt_t2": float(t2_opt),
        "ABCD/nonclosure": float(best["nonclosure"]),
        "ABCD/A": int(best["A"]),
        "ABCD/B": int(best["B"]),
        "ABCD/C": int(best["C"]),
        "ABCD/D": int(best["D"]),
    })

    #########
    #PLOTTING
    #########

    #2D histogram bkg only
    fig = plt.figure(figsize=(6,5))
    plt.hist2d(reco_test_m, axis2_bkg_m, bins=200, norm=LogNorm(vmin=1), cmin=1)
    plt.axvline(t1_opt, color="black", ls="--")
    plt.axhline(t2_opt, color="black", ls="--")
    plt.xscale("log")
    plt.xlabel("AE loss")
    plt.ylabel("Contrastive score")
    plt.title("AE vs Contrastive (bkg only)")
    out = os.path.join(plot_dir, "hist2d_bkg.png")
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.close()
    wandb.log({"Hists2D/bkg": wandb.Image(out)})

    #2D histrogram bkg and sig
    fig, ax = plt.subplots(figsize=(6,5))
    hb = ax.hist2d(reco_test_m, axis2_bkg_m, bins=200, norm=LogNorm(vmin=1), cmin=1)
    hs = ax.hist2d(reco_sig_m,  axis2_sig_m, bins=200, norm=LogNorm(vmin=1), cmin=1, alpha=0.65)
    ax.axvline(t1_opt, color="black", ls="--", lw=1.5)
    ax.axhline(t2_opt, color="black", ls="--", lw=1.5)
    ax.set_xscale("log")
    ax.set_xlabel("AE loss")
    ax.set_ylabel("Contrastive score")
    ax.set_title("AE vs Contrastive")
    fig.colorbar(hb[3], ax=ax, pad=0.01, label="Background counts")
    fig.colorbar(hs[3], ax=ax, pad=0.08, label="Signal counts")
    out_overlay = os.path.join(plot_dir, "hist2d_overlay.png")
    fig.savefig(out_overlay, dpi=200, bbox_inches="tight")
    plt.close(fig)
    wandb.log({"Hists2D/bkg_sig": wandb.Image(out_overlay)})

    #profile plots
    fig, ax = plt.subplots(figsize=(8, 6))
    #can't do logx for this because contrastive loss could be negative?
    profile_plot(ax, axis2_bkg_m, reco_test_m, nbins=60, logx=False)
    ax.set_xlabel("Contrastive score")
    ax.set_ylabel("Mean AE loss")
    ax.set_yscale("log")
    ax.set_title("⟨AE loss⟩ vs contrastive")
    p1_path = os.path.join(plot_dir, "profile_AE_vs_contrastive.png")
    fig.savefig(p1_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    wandb.log({"Profiles/AE_vs_contrastive": wandb.Image(p1_path)})

    fig, ax = plt.subplots(figsize=(8, 6))
    profile_plot(ax, reco_test_m, axis2_bkg_m, nbins=60, logx=True)
    ax.set_xlabel("AE loss")
    ax.set_ylabel("Mean contrastive score")
    ax.set_title("⟨contrastive⟩ vs AE loss")
    p2_path = os.path.join(plot_dir, "profile_contrastive_vs_AE.png")
    fig.savefig(p2_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    wandb.log({"Profiles/contrastive_vs_AE": wandb.Image(p2_path)})
    
    #1D scan just for plotting closure and S/sqrt(B)
    effs = []
    closure_ratio = []
    closure_unc = []
    s_over_sqrtb = []
    Ntot_bkg = float(len(reco_test_m))

    for p in percent:
        t1, t2, A, B, C, D = abcd_counts(reco_test_m, axis2_bkg_m, p, p)
        A_hat = (B*C)/max(D, 1e-8)
        ratio = A_hat/max(A, 1e-8)

        invA = 0.0 if A == 0 else 1.0/A
        invB = 0.0 if B == 0 else 1.0/B
        invC = 0.0 if C == 0 else 1.0/C
        invD = 0.0 if D == 0 else 1.0/D
        rel_var = invA + invB + invC + invD
        sigma = abs(ratio)*np.sqrt(rel_var) if rel_var > 0 else 0.0

        effs.append(A/max(Ntot_bkg, 1.0))
        closure_ratio.append(ratio)
        closure_unc.append(sigma)

        A_sig = int(((reco_sig_m > t1) & (axis2_sig_m > t2)).sum())
        s_over_sqrtb.append(A_sig/np.sqrt(max(A, 1e-8)))

    effs = np.array(effs)
    closure_ratio = np.array(closure_ratio)
    closure_unc = np.array(closure_unc)
    s_over_sqrtb = np.array(s_over_sqrtb)

    #order everything for plotting
    order = np.argsort(effs)
    effs = effs[order]
    closure_ratio = closure_ratio[order]
    closure_unc = closure_unc[order]
    s_over_sqrtb = s_over_sqrtb[order]

    #compute optimized eff and S/sqrt(B) for putting on the plot
    eff_opt = best["A"]/max(Ntot_bkg, 1.0)
    ratio_opt = best["A_hat"]/max(best["A"], 1e-8)
    sigA_opt = int(((reco_sig_m > t1_opt) & (axis2_sig_m > t2_opt)).sum())
    s_over_sqrtb_opt = sigA_opt/np.sqrt(max(best["A"], 1e-8))

    #closure and s/sqrt(b) plots
    
    colors = ['g', 'b']
    fig_size = (8, 6)
    fs = 28
    fs_leg = 24
    
    fig, ax = plt.subplots(figsize=fig_size)
    
    # main curve
    ax.plot(effs, closure_ratio, c=colors[0], label="AE + Contrastive")
    
    # uncertainty band
    alpha_band = 0.5
    low = closure_ratio - closure_unc
    high = closure_ratio + closure_unc
    
    ax.fill_between(effs, low, high, facecolor=colors[0], alpha=alpha_band, interpolate=True)
    one = np.ones_like(effs)
    one_m = np.full_like(effs, 0.95)
    one_p = np.full_like(effs, 1.05)
    ax.plot(effs, one, linestyle='-',  color='black')
    ax.plot(effs, one_m, linestyle='--', color='black')
    ax.plot(effs, one_p, linestyle='--', color='black')
    ax.plot([eff_opt], [ratio_opt], marker='o', c='red', label='Optimized')
    ax.set_xlabel('Selection Efficiency (bkg A/Ntot)', fontsize=fs)
    ax.set_ylabel('Predicted Bkg. / True Bkg.', fontsize=fs)
    plt.ylim([0.0, 1.5])
    plt.xscale('log')
    plt.tick_params(axis='x', labelsize=fs_leg)
    plt.tick_params(axis='y', labelsize=fs_leg)
    plt.legend(loc="lower right", fontsize=fs_leg)
    closure_path = os.path.join(plot_dir, "cut_and_count_bkg_check.png")
    plt.savefig(closure_path, dpi=200, bbox_inches='tight')
    plt.close()
    wandb.log({"Closure/plot": wandb.Image(closure_path)})
    
    
    #S/sqrt(B) plot   
    fig, ax = plt.subplots(figsize=fig_size)
    ax.plot(effs, s_over_sqrtb, color="red", label=r"$S/\sqrt{B}$")
    ax.plot([eff_opt], [s_over_sqrtb_opt], marker='o', color='black')
    ax.set_xlabel('Selection Efficiency (bkg A/Ntot)', fontsize=fs)
    ax.set_ylabel(r"$S/\sqrt{B}$", fontsize=fs)
    plt.xscale('log')
    plt.tick_params(axis='x', labelsize=fs_leg)
    plt.tick_params(axis='y', labelsize=fs_leg)
    plt.legend(loc="best", fontsize=fs_leg)
    sig_path = os.path.join(plot_dir, "s_over_sqrtb_vs_bkg_eff.png")
    plt.savefig(sig_path, dpi=200, bbox_inches='tight')
    plt.close()
    wandb.log({"Signal/s_over_sqrtb_vs_bkg_eff": wandb.Image(sig_path)})

    return dict(
        ae=ae,
        reco_test=reco_test_m,
        reco_sig=reco_sig_m,
        axis2_bkg=axis2_bkg_m,
        axis2_sig=axis2_sig_m,
        thresholds=(t1_opt, t2_opt),
        best=best,
        plots=[out, out_overlay, closure_path, sig_path, p1_path, p2_path],
        ae_path=ae_path,
    )
    

In [12]:
config = {
  "data_path": "/axovol/double_disco/HLT_scout_bkgd_2024I.h5",
  "sig_path": "/axovol/double_disco/signal_tptp.h5",
  "contrast_bkg_scores": "/axovol/double_disco/contrastive_scores_bkg_test.npy",
  "contrast_sig_scores": "/axovol/double_disco/contrastive_scores_sig.npy",
  "ae_lr": 1e-4,
  "alpha": 0.5,
  "ae_latent": 16,
  "enc_nodes": [128, 64, 32],
  "dec_nodes": [32, 64, 128],
  "epochs": 50,
  "batch_size": 2048,
  "min_A": 200,
  "min_D": 1000,
  "outdir": "ae_vs_contrastive_abcd",  
}

In [ ]:
results = run(config)